In [ ]:
from __future__ import annotations

import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import manifold_dynamics.tuning_utils as tut
import manifold_dynamics.neural_utils as nu
import manifold_dynamics.paths as pth
import manifold_dynamics.spike_response_stats as srs
import visionlab_utils.storage as vst

In [ ]:
# MAIN IDEA: test for cross-validated subspace rotation
# HYPOTHESIS: transient subspace rotations that depend on location/scale on the natural image manifold
# CONCRETE: represent each time bin by the top-d PCs of the image-response matrix, then measure how much those subspaces rotate across time using principal angles

# (units, time, images, reps)
# images = either (1) local/top-k manifold scale or (2) all images


# ROI format:
# - 4-part UID: SesIdx.RoiIndex.AREALABEL.Categoty (e.g., 18.19.Unknown.F)
# - 3-part ROI: RoiIndex.AREALABEL.Categoty (e.g., 19.Unknown.F)
ROI_TARGET = "19.Unknown.F"
ALPHA = 0.05
BIN_SIZE_MS = 20
VERBOSE = True

def vprint(msg: str) -> None:
    if VERBOSE:
        print(msg)

uid_csv_uri = f"{pth.OTHERS}/roi-uid.csv"
uid_csv_local = vst.fetch(uid_csv_uri)
df_uid = pd.read_csv(uid_csv_local)

CAT = "face"
with open(f'./../datasets/NNN/{CAT}_mins.pkl', 'rb') as f:
    mins = pickle.load(f)
    
SCALES = {k: v[0] for k,v in mins.items()}
scale = SCALES['Unknown_19_F']

In [ ]:
target_parts = ROI_TARGET.split(".")
if len(target_parts) not in (3, 4):
    raise ValueError(
        "ROI_TARGET must be either 4-part UID (SesIdx.RoiIndex.AREALABEL.Categoty) "
        "or 3-part ROI (RoiIndex.AREALABEL.Categoty)."
    )

if len(target_parts) == 4:
    roi_uids = [ROI_TARGET]
else:
    roi_index = int(target_parts[0])
    area_label = target_parts[1]
    category = target_parts[2]
    roi_uids = []
    for uid in df_uid["uid"].astype(str):
        parts = uid.split(".")
        if len(parts) != 4:
            continue
        if int(parts[1]) == roi_index and parts[2] == area_label and parts[3] == category:
            roi_uids.append(uid)
    roi_uids = sorted(roi_uids, key=lambda x: int(x.split(".")[0]))

if len(roi_uids) == 0:
    raise ValueError(f"No matching ROI UIDs found for target: {ROI_TARGET}")

vprint(f"Resolved ROI target {ROI_TARGET} to UIDs: {roi_uids}")


# Final tensors:
#   psth_A: (units, time, images) from trials [0,2,4,...]
#   psth_B: (units, time, images) from trials [1,3,5,...]
parts_A: list[np.ndarray] = []
parts_B: list[np.ndarray] = []
units_per_uid: dict[str, int] = {}

for uid in roi_uids:
    raster_4d = nu.load_cached_session_raster(uid)
    raster_4d = nu.bin_to_psth(raster_4d, bin_size_ms=BIN_SIZE_MS)
    vprint(f"{uid}: raster shape after {BIN_SIZE_MS}ms binning = {raster_4d.shape}")

    pvals = srs.is_responsive(X=raster_4d, roi_uid=uid, test_type="paired").squeeze()
    responsive_mask = np.isfinite(pvals) & (pvals < ALPHA)
    n_responsive = int(np.sum(responsive_mask))
    if n_responsive == 0:
        vprint(f"{uid}: no responsive units at alpha={ALPHA}; skipping")
        continue

    split_a = raster_4d[:, :, :, 0::2]
    split_b = raster_4d[:, :, :, 1::2]

    psth_a_uid = np.nanmean(split_a, axis=3)[responsive_mask]  # (units, time, images)
    psth_b_uid = np.nanmean(split_b, axis=3)[responsive_mask]  # (units, time, images)

    parts_A.append(psth_a_uid)
    parts_B.append(psth_b_uid)
    units_per_uid[uid] = n_responsive

    vprint(
        f"{uid}: responsive={n_responsive}, "
        f"A shape={psth_a_uid.shape}, B shape={psth_b_uid.shape}"
    )

if len(parts_A) == 0 or len(parts_B) == 0:
    raise ValueError("No responsive units across selected ROI/session(s).")

psth_A = np.concatenate(parts_A, axis=0)
psth_B = np.concatenate(parts_B, axis=0)

if psth_A.shape != psth_B.shape:
    raise ValueError(f"A/B shape mismatch: A={psth_A.shape}, B={psth_B.shape}")

vprint(f"Combined responsive units by session: {units_per_uid}")
vprint(f"Final PSTH A shape (units, time, images): {psth_A.shape}")
vprint(f"Final PSTH B shape (units, time, images): {psth_B.shape}")

In [ ]:
rasters = []
for uid in roi_uids:
    raster_4d = nu.load_cached_session_raster(uid)
    raster_4d = nu.bin_to_psth(raster_4d, bin_size_ms=BIN_SIZE_MS)
    raster_3d = np.nanmean(raster_4d, axis=3)
    rasters.append(raster_3d)
raster_3d = np.vstack(rasters)

pvals = srs.is_responsive(raster_3d, uid, test_type="paired").squeeze()
responsive_mask = np.isfinite(pvals) & (pvals < ALPHA)
train_3d = raster_3d[responsive_mask]

order = tut.rank_images_by_response(train_3d)

In [ ]:
def pca_subspace_basis(X_u_by_img: np.ndarray, d: int, center: bool = True, eps: float = 1e-12) -> np.ndarray:
    """
    X_u_by_img: (units, images)
    Returns Q: (units, d) orthonormal PCA basis (top-d left singular vectors).
    """
    X = X_u_by_img.astype(np.float64, copy=False)

    if center:
        mu = np.nanmean(X, axis=1, keepdims=True)
        X = X - mu

    # Replace any remaining NaNs with 0 after centering (safe if NaNs are sparse)
    X = np.nan_to_num(X, nan=0.0)

    # SVD on (units x images); left singular vecs are PCA directions in unit space
    U, S, Vt = np.linalg.svd(X, full_matrices=False)

    # Effective rank and d clamp
    r = int(np.sum(S > eps))
    dd = int(min(d, r, U.shape[1]))
    if dd == 0:
        # Fallback: empty basis if all-zero / rank-0
        return np.zeros((U.shape[0], 0), dtype=np.float64)

    Q = U[:, :dd]
    return Q

def principal_angles(Q1: np.ndarray, Q2: np.ndarray) -> np.ndarray:
    """
    Q1: (units, d1) orthonormal columns
    Q2: (units, d2) orthonormal columns
    Returns angles (min(d1,d2),) in radians.
    """
    if Q1.size == 0 or Q2.size == 0:
        return np.array([], dtype=np.float64)

    M = Q1.T @ Q2
    s = np.linalg.svd(M, compute_uv=False)
    s = np.clip(s, 0.0, 1.0)
    return np.arccos(s)

In [ ]:
scales = [scale, 1072]
d = 10  # top d PCs
T = psth_A.shape[1]

# windows (edit as needed)
base_sl = slice(0, 50)
post_sl = slice(160, 200)

fig, axes = plt.subplots(
    2, 2, figsize=(10, 8),
    gridspec_kw={"width_ratios": [1.35, 1.0]}
)

for r, sc in enumerate(scales):
    image_idx = order[:sc]

    QA, QB = [], []
    for t in range(T):
        XA = psth_A[:, t, image_idx]  # (units, images_subset)
        XB = psth_B[:, t, image_idx]
        QA.append(pca_subspace_basis(XA, d=d))
        QB.append(pca_subspace_basis(XB, d=d))

    sim_cv = np.full((T, T), np.nan, dtype=np.float64)
    for t1 in range(T):
        QAt1, QBt1 = QA[t1], QB[t1]
        for t2 in range(T):
            QAt2, QBt2 = QA[t2], QB[t2]

            ang1 = principal_angles(QAt1, QBt2)
            ang2 = principal_angles(QBt1, QAt2)
            
            s1 = float(np.mean(np.cos(ang1) ** 2))
            s2 = float(np.mean(np.cos(ang2) ** 2))
            
            # symmetrized cross-validated distance
            # dist_cv = np.nanmean([d1, d2])
            sim_cv[t1, t2] = np.nanmean([s1, s2])

    # --- left column: matrices ---
    ax_hm = axes[r, 0]
    sns.heatmap(sim_cv, square=True, ax=ax_hm, cbar=(r == 0))
    ax_hm.set_title(f"{sc} images")
    ax_hm.set_xlabel("time")
    ax_hm.set_ylabel("time")

    # --- right column: baseline vs post bar plot ---
    base_mean = float(np.nanmean(sim_cv[base_sl, base_sl]))
    post_mean = float(np.nanmean(sim_cv[post_sl, post_sl]))
    base_diag_mean = float(np.nanmean(np.diag(sim_cv[base_sl, base_sl])))
    post_diag_mean = float(np.nanmean(np.diag(sim_cv[post_sl, post_sl])))

    ax_bar = axes[r, 1]
    ax_bar.bar(["baseline", "post", "diag base", "diag post"], [base_mean, post_mean, base_diag_mean, post_diag_mean])
    # ax_bar.set_ylim(0, 1)
    ax_bar.set_title("mean(sim_cv)")

    # optional: annotate
    ax_bar.text(0, base_mean, f"{base_mean:.3f}", ha="center", va="bottom")
    ax_bar.text(1, post_mean, f"{post_mean:.3f}", ha="center", va="bottom")
    ax_bar.text(2, post_mean, f"{base_diag_mean:.3f}", ha="center", va="bottom")
    ax_bar.text(3, post_mean, f"{post_diag_mean:.3f}", ha="center", va="bottom")


plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(10, 4), gridspec_kw={"width_ratios": [1.35, 1.0]})

d = 10 # top d PCs
T = psth_A.shape[1]
QA = []
QB = []

rng = np.random.default_rng(0)
image_idx = rng.choice(order[:scale], scale)

for t in range(T):
    XA = psth_A[:, t, image_idx]  # (units, images_subset)
    XB = psth_B[:, t, image_idx]
    QA.append(pca_subspace_basis(XA, d=d))
    QB.append(pca_subspace_basis(XB, d=d))

sim_cv = np.full((T, T), np.nan, dtype=np.float64)
for t1 in range(T):
    QAt1, QBt1 = QA[t1], QB[t1]
    for t2 in range(T):
        QAt2, QBt2 = QA[t2], QB[t2]

        ang1 = principal_angles(QAt1, QBt2)
        ang2 = principal_angles(QBt1, QAt2)
        
        s1 = float(np.mean(np.cos(ang1) ** 2))
        s2 = float(np.mean(np.cos(ang2) ** 2))
        
        # symmetrized cross-validated distance
        # dist_cv = np.nanmean([d1, d2])
        sim_cv[t1, t2] = np.nanmean([s1, s2])

ax = axes[0]
sns.heatmap(sim_cv, square=True, ax=ax)
ax.set_title(f'Random {scale} images')

base_mean = float(np.nanmean(sim_cv[base_sl, base_sl]))
post_mean = float(np.nanmean(sim_cv[post_sl, post_sl]))
base_diag_mean = float(np.nanmean(np.diag(sim_cv[base_sl, base_sl])))
post_diag_mean = float(np.nanmean(np.diag(sim_cv[post_sl, post_sl])))

ax_bar = axes[1]
ax_bar.bar(["baseline", "post", "diag base", "diag post"], [base_mean, post_mean, base_diag_mean, post_diag_mean])
# ax_bar.set_ylim(0, 1)
ax_bar.set_title("mean(sim_cv)")

# optional: annotate
ax_bar.text(0, base_mean, f"{base_mean:.3f}", ha="center", va="bottom")
ax_bar.text(1, post_mean, f"{post_mean:.3f}", ha="center", va="bottom")
ax_bar.text(2, post_mean, f"{base_diag_mean:.3f}", ha="center", va="bottom")
ax_bar.text(3, post_mean, f"{post_diag_mean:.3f}", ha="center", va="bottom")
plt.tight_layout()